# Session 2 · Writing Python that does not break
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/neoma-python-2627/blob/main/notebooks/S02_lecture_robust_python.ipynb)  ·  *Python Programming for Business Analytics · NEOMA 2026/27 · dr. Uroš Godnov*

**Live on Zoom · 3 hours.** Wrong numbers that look right are the most expensive bug in business analytics. Today your code learns to **fail loudly, explain why, and keep going when it should**.

| Part | Time | Topic |
|---|---|---|
| A1 | 0:00 | Reading a traceback |
| A2 | 0:15 | Functions with guard rails |
| A3 | 0:45 | Exceptions: `try`, `except`, your own error classes, warnings |
| A4 | 1:15 | 🤖 AI moment: explain this error |
| ☕ | **1:30** | **Break · 10 minutes** |
| B1 | 1:40 | Files and iteration with `pathlib` and comprehensions |
| B2 | 2:05 | Case: the bullet-proof importer for seven sales files |
| B3 | 2:50 | Wrap-up |

🆚 This is the Python version of *Writing R that does not break* (`cli_abort()`, `tryCatch()`, `purrr::safely()`).

In [ ]:
# ⚙️ Setup: run this cell first (works in Google Colab and in Positron)
from pathlib import Path
import numpy as np
import pandas as pd

REPO = "https://raw.githubusercontent.com/urosgodnov/neoma-python-2627/main/"


def data_path(name):
    """Use the local course folder if you have it, otherwise read the file from GitHub."""
    for base in (Path("data"), Path("../data")):
        if (base / name).exists():
            return base / name
    return REPO + "data/" + name


pd.set_option("display.max_columns", 30)
print("Python ready · pandas", pd.__version__)
import warnings

# The seven monthly sales files (identical to the R course). In Colab they are downloaded once.
SALES_DIR = next((p for p in (Path("data/sales"), Path("../data/sales")) if p.exists()), None)
if SALES_DIR is None:
    import urllib.request
    SALES_DIR = Path("data/sales")
    SALES_DIR.mkdir(parents=True, exist_ok=True)
    for m in range(1, 8):
        name = f"sales_2026_{m:02d}.csv"
        urllib.request.urlretrieve(REPO + "data/sales/" + name, SALES_DIR / name)
print(sorted(p.name for p in SALES_DIR.glob("*.csv")))

---
## A1 · Reading a traceback

A **traceback** is Python's error report. Read it **from the bottom up**: the last line says *what* went wrong, the lines above say *where*.

In [ ]:
def revenue(units, unit_price):
    return units * unit_price


def monthly_total(orders):
    return sum(revenue(o["units"], o["price"]) for o in orders)


orders = [{"units": 3, "price": 9.9}, {"units": "2", "price": 4.5}]
try:
    monthly_total(orders)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

🔮 **Predict:** remove the `try/except` (run `monthly_total(orders)` alone). Which line does the arrow point to? Which order caused the error?

The most common exceptions you will meet:

| Exception | Typical cause |
|---|---|
| `NameError` | typo in a variable name, or a cell you did not run |
| `TypeError` | wrong type: `"2" * 4.5`, calling a number |
| `ValueError` | right type, wrong value: `float("twelve")` |
| `KeyError` | column or dictionary key does not exist: `df["Unit Price"]` |
| `IndexError` | position does not exist: `x[10]` in a list of 5 |
| `FileNotFoundError` | wrong path |
| `ZeroDivisionError` | `x / 0` with plain numbers (pandas returns `inf` instead!) |

In [ ]:
monthly_total([{"units": 3, "price": 9.9}])

---
## A2 · Functions with guard rails

A function should **reject bad input at the door**, with a message that tells the user what to fix. `raise` is Python's `cli_abort()`.

In [ ]:
def discount_price(price, pct):
    """Price after a percentage discount, rounded to 2 decimals."""
    if not isinstance(price, (int, float)) or isinstance(price, bool):
        raise TypeError(f"`price` must be a number, not {type(price).__name__} ({price!r}).")
    if not 0 <= pct <= 100:
        raise ValueError(f"`pct` must be between 0 and 100. You supplied {pct!r}.")
    return round(price * (1 - pct / 100), 2)


discount_price(80, 25)

In [ ]:
for bad_call in [lambda: discount_price("80", 25), lambda: discount_price(80, 250)]:
    try:
        bad_call()
    except (TypeError, ValueError) as e:
        print(f"❌ {type(e).__name__}: {e}")

**Allowed values only** (Python's `rlang::arg_match()`): check membership in a set and list the options in the message.

In [ ]:
SHIPPING = {"standard": 4.90, "express": 9.90, "pickup": 0.0}


def shipping_cost(method):
    if method not in SHIPPING:
        raise ValueError(f"`method` must be one of {sorted(SHIPPING)}, not {method!r}.")
    return SHIPPING[method]


try:
    shipping_cost("Express")
except ValueError as e:
    print("❌", e)

💬 **Discuss:** should `shipping_cost("Express")` quietly lower-case the input, or fail? Who decides?

⚠️ `assert` is for **your own** sanity checks during development. It can be switched off, so never use it to validate user input.

🧪 **Try it (4 min):** write `margin_pct(price, cost)` that returns the margin in % and raises a `ValueError` with a clear message if `price` is not positive.

In [ ]:
# your code here

---
## A3 · Exceptions: `try`, `except`, your own error classes, warnings

```python
try:
    risky()            # code that may fail
except ValueError as e:
    handle(e)          # only this error type
else:
    on_success()       # runs if no error
finally:
    cleanup()          # runs always
```
🆚 R: `tryCatch(expr, error = function(e) ..., finally = ...)`.

In [ ]:
def to_number(text):
    try:
        value = float(text)
    except ValueError:
        print(f"   ⚠️ could not convert {text!r}, using NaN")
        return np.nan
    else:
        return value


[to_number(t) for t in ["12", "7.5", "twelve", "3"]]

### Your own error classes (classed conditions)
A custom class lets the **caller** decide which problems to handle and which to let through. 🆚 `cli_abort(class = "zero_division_error")`.

In [ ]:
class ZeroDenominatorError(ValueError):
    """Raised when a business ratio has a zero denominator."""


def safe_ratio(a, b):
    if b == 0:
        raise ZeroDenominatorError(f"Cannot divide {a} by zero.")
    return a / b


def conversion_rate(orders, visits):
    try:
        return safe_ratio(orders, visits)
    except ZeroDenominatorError as e:
        print(f"ℹ️ {e} Returning NaN.")
        return np.nan          # this specific problem is handled ...
    # ... any other error (e.g. TypeError) is NOT caught and stops the program


conversion_rate(12, 400), conversion_rate(0, 0)

🔮 **Predict:** what does `conversion_rate("12", 400)` do? Handled or not?

### Warnings: keep going, but tell me
A **warning** does not stop the code. You can **collect** warnings instead of printing them (🆚 `withCallingHandlers()`).

In [ ]:
def parse_units(values):
    out = []
    for v in values:
        try:
            out.append(int(v))
        except ValueError:
            warnings.warn(f"units value {v!r} is not a whole number")
            out.append(np.nan)
    return out


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    units = parse_units(["12", "7", "n/a", "3", "twelve"])

print(units)
print([str(w.message) for w in caught])

In pandas the same job is one line, but **count what you lost**:

In [ ]:
raw = pd.Series(["12", "7", "n/a", "3", "twelve"])
num = pd.to_numeric(raw, errors="coerce")
print(num.tolist())
print(f"{num.isna().sum() - raw.isna().sum()} values became NaN during conversion: {raw[num.isna()].tolist()}")

---
## A4 · 🤖 AI moment: explain this error

Run the cell below. In **Colab**, click *Explain error* (Gemini). In **Positron**, select the error and ask the Assistant *"Explain this error and suggest a fix"*. Then fix it yourself.

💬 Was the AI right? Did it fix the **cause** or only the **symptom**?

In [ ]:
prices = {"Notebook": 4.9, "Pen set": 12.5}
basket = ["Notebook", "Planner"]
try:
    total = sum(prices[item] for item in basket)
except KeyError as e:
    print("KeyError:", e)

---
## ☕ Break · 10 minutes
---

## B1 · Files and iteration

`pathlib.Path` is the modern way to work with files. It works on Windows, macOS and Linux.

In [ ]:
sales_files = sorted(SALES_DIR.glob("*.csv"))
[f.name for f in sales_files]

In [ ]:
sizes_kb = {f.name: round(f.stat().st_size / 1024, 2) for f in sales_files}   # dict comprehension
sizes_kb

🔮 **Predict:** one file is suspicious just from its size. Which one, and what will happen when we read it?

Read several files and stack them, keeping the file name. 🆚 `map(read_csv) |> list_rbind(names_to = "file")`.

In [ ]:
picked = [sales_files[i] for i in (0, 1, 4)]           # January, February, May
combined = pd.concat({f.name: pd.read_csv(f) for f in picked}, names=["file", "row"]).reset_index("file")
combined.head()

In [ ]:
(
    combined
    .assign(revenue=lambda d: d["units"] * d["unit_price"])
    .groupby(["file", "region"], as_index=False)["revenue"].sum()
    .pivot(index="region", columns="file", values="revenue")
    .round(0)
)

`zip()` walks over several lists in parallel. 🆚 `map2()`.

In [ ]:
net_prices = [10, 20, 30]
vat_rates = [0.20, 0.10, 0.055]
print([round(p * (1 + r), 2) for p, r in zip(net_prices, vat_rates)])
print((np.array(net_prices) * (1 + np.array(vat_rates))).round(2))   # vectorised: no loop at all

---
## B2 · Case: the bullet-proof importer

Finance drops a CSV file into a folder every month. Your report must read **all** of them, every month, without anyone checking by hand. Let's see what reality looks like.

In [ ]:
REQUIRED = ["order_id", "order_date", "region", "product", "units", "unit_price"]

for f in sales_files:
    first_line = f.read_text(encoding="utf-8").splitlines()[:1]
    print(f"{f.name}: {first_line}")

💬 **Discuss:** list every problem you can see. Which ones should **stop** the import, and which ones can we **repair**?

### Version 1 · a strict reader

In [ ]:
class SalesFileError(Exception):
    """A sales file cannot be used."""


def read_sales_strict(path):
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        raise SalesFileError("file is empty") from None
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise SalesFileError(f"missing columns {missing}")
    return df


for f in sales_files:
    try:
        df = read_sales_strict(f)
        print(f"✅ {f.name}: {len(df)} rows")
    except SalesFileError as e:
        print(f"❌ {f.name}: {e}")

### Version 2 · repair what is safe to repair, reject the rest
- semicolon files use a **decimal comma**: detect the separator from the header
- header spelling (`Unit Price`): normalise to `snake_case`
- a missing column or an empty file: **reject**, with the reason

In [ ]:
def clean_names(columns):
    return [c.strip().lower().replace(" ", "_") for c in columns]


def read_sales(path):
    """Read one monthly sales file. Raise SalesFileError if it cannot be used."""
    path = Path(path)
    header = path.read_text(encoding="utf-8").strip().splitlines()[:1]
    if not header:
        raise SalesFileError("file is empty")
    european = ";" in header[0]
    df = pd.read_csv(
        path,
        sep=";" if european else ",",
        decimal="," if european else ".",
        skipinitialspace=True,
        dtype={"units": "string"},          # read as text, convert deliberately below
    )
    df.columns = clean_names(df.columns)
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise SalesFileError(f"missing columns {missing}")
    return df


read_sales(sales_files[2]).head(3)

### The import report
🆚 `safely()`: wrap the reader so that one bad file never stops the loop, and keep a record of what happened.

In [ ]:
def safely(func):
    """Return a version of func that never raises: (result, error)."""
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs), None
        except Exception as e:                 # broad on purpose: we record, we do not hide
            return None, e
    return wrapper


safe_read = safely(read_sales)
results = {f.name: safe_read(f) for f in sales_files}

import_report = pd.DataFrame(
    {
        "file": name,
        "ok": err is None,
        "rows": 0 if df is None else len(df),
        "error_message": "" if err is None else str(err),
    }
    for name, (df, err) in results.items()
)
import_report

In [ ]:
for row in import_report.itertuples():
    if row.ok:
        print(f"✅ {row.file}: {row.rows} rows imported")
    else:
        print(f"❌ {row.file}: {row.error_message}")

### Combine, convert, and count the damage

In [ ]:
ok_frames = {name: df for name, (df, err) in results.items() if err is None}
sales = pd.concat(ok_frames, names=["file", "row"]).reset_index("file")

units_num = pd.to_numeric(sales["units"], errors="coerce")
lost = sales.loc[units_num.isna() & sales["units"].notna(), ["file", "order_id", "units"]]
sales = sales.assign(
    units=units_num.astype("Int64"),
    unit_price=sales["unit_price"].astype(float),
    order_date=pd.to_datetime(sales["order_date"]),
)
print(f"{len(sales)} orders from {sales['file'].nunique()} files · {len(lost)} unit values became missing:")
lost

In [ ]:
sales.dtypes

💬 **Discuss:** `"twelve"` became missing. Should we turn it into `12`? Who should fix it: you, or the colleague who produces the file?

In [ ]:
(
    sales
    .assign(revenue=lambda d: d["units"] * d["unit_price"], month=lambda d: d["order_date"].dt.month)
    .groupby("month")["revenue"].sum()
    .round(0)
)

---
## B3 · Wrap-up

| Idea | Python | R |
|---|---|---|
| Stop with a message | `raise ValueError(...)` | `cli_abort()` |
| Handle an error | `try / except` | `tryCatch()` |
| Your own error type | `class MyError(Exception)` | `cli_abort(class = ...)` |
| Collect warnings | `warnings.catch_warnings(record=True)` | `withCallingHandlers()` |
| Never crash the loop | `safely()` wrapper (or `try` inside the loop) | `purrr::safely()` |
| Iterate over files | `Path.glob()` + comprehension | `list.files()` + `map()` |

**Before Session 3:** practice notebook **S02_practice** and the **Session 2 quiz**.

💬 **Exit question:** which error in your own past work would this approach have caught?